# 0. Problem
## 602. Friend Requests II: Who Has the Most Friends — Medium
Treat both requester and accepter as friendship endpoints. Return the person with the largest total number of accepted friends as `id, num`.

Official: https://leetcode.com/problems/friend-requests-ii-who-has-the-most-friends/

# 1. Setup

In [ ]:
import pandas as pd
requests_rows=[(1,2,"2016/06/03"),(1,3,"2016/06/08"),(2,3,"2016/06/08"),(3,4,"2016/06/09")]
requests_pd=pd.DataFrame(requests_rows,columns=["requester_id","accepter_id","accept_date"])
requests_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
requests_spark=spark.createDataFrame(requests_rows,["requester_id","accepter_id","accept_date"])
requests_spark.createOrReplaceTempView("RequestAccepted")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
WITH people AS (
    SELECT requester_id AS id FROM RequestAccepted
    UNION ALL
    SELECT accepter_id AS id FROM RequestAccepted
)
SELECT id, COUNT(*) AS num
FROM people
GROUP BY id
ORDER BY num DESC, id ASC
LIMIT 1
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
people=pd.concat([requests_pd["requester_id"],requests_pd["accepter_id"]],ignore_index=True).rename("id")
counts=people.value_counts().rename_axis("id").reset_index(name="num")
result_pd=counts.sort_values(["num","id"],ascending=[False,True]).head(1).reset_index(drop=True)
result_pd

# 4. PySpark Solution

In [ ]:
people=requests_spark.select(F.col("requester_id").alias("id")).unionAll(requests_spark.select(F.col("accepter_id").alias("id")))
result_spark=(people.groupBy("id").agg(F.count("*").alias("num")).orderBy(F.desc("num"),F.asc("id")).limit(1))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| normalize two endpoint columns | `UNION ALL` | `pd.concat()` | `.unionAll()` |
| frequency | `GROUP BY + COUNT` | `.value_counts()` | `.groupBy().count()` |
| top row | `ORDER BY ... LIMIT 1` | `.sort_values().head(1)` | `.orderBy().limit(1)` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): RequestAccepted

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: requests_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: requests_spark